# Processing

Explore the oral cavity images in `artifacts/raw_data/` and inspect the preprocessing that `pipeline/data_preprocessing.py` applies before inference: resize to 224x224, per-channel CLAHE, scale to [0, 1].

In [ ]:
import sys
from pathlib import Path

BACKEND_ROOT = Path.cwd().parents[2]
sys.path[:0] = [str(BACKEND_ROOT), str(BACKEND_ROOT / "src")]

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from prayaas.config.configuration import CLASS_LABELS, settings

settings.raw_data_dir

## Index the class folders

In [ ]:
from pipeline.data_ingestion import DataIngestion

index = DataIngestion().run()
print(index["folder"].value_counts(), "\n")
index.head()

## Class balance

500 CANCER vs 200 NON CANCER — a 2.5:1 imbalance. Accuracy alone will flatter a model that leans toward CANCER, so read per-class recall instead.

In [ ]:
index["folder"].value_counts().plot(kind="bar", title="Images per class")
plt.tight_layout()
plt.show()

## Raw image dimensions

In [ ]:
sizes = []
for path in index["path"].sample(min(60, len(index)), random_state=0):
    with Image.open(path) as im:
        sizes.append(im.size)

widths, heights = zip(*sizes)
print(f"width  {min(widths)}-{max(widths)}")
print(f"height {min(heights)}-{max(heights)}")
print(f"\nAll are resized to {settings.image_size}x{settings.image_size} before inference.")

## Sample images per class

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 7))

for row, folder in enumerate(index["folder"].unique()):
    paths = index[index["folder"] == folder]["path"].head(4)
    for col, path in enumerate(paths):
        with Image.open(path) as im:
            axes[row, col].imshow(im)
        axes[row, col].set_title(folder, fontsize=9)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## What CLAHE does

Contrast Limited Adaptive Histogram Equalisation, applied per channel. The original Streamlit client ran this before upload, so the pipeline reproduces it server-side to keep inputs consistent. Toggle with `APPLY_CLAHE` in `.env`.

In [ ]:
from pipeline.data_preprocessing import apply_clahe

sample_path = index["path"].iloc[0]
with Image.open(sample_path) as im:
    original = np.array(im.convert("RGB").resize((224, 224)))

enhanced = apply_clahe(original)

fig, axes = plt.subplots(1, 2, figsize=(9, 5))
axes[0].imshow(original); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(enhanced); axes[1].set_title("CLAHE"); axes[1].axis("off")
plt.tight_layout()
plt.show()

## Channel histograms, before and after

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for image, ax, title in ((original, axes[0], "Original"), (enhanced, axes[1], "CLAHE")):
    for channel, colour in enumerate("rgb"):
        ax.hist(image[:, :, channel].ravel(), bins=64, alpha=0.5, color=colour)
    ax.set_title(title)

plt.tight_layout()
plt.show()

## The exact array the model receives

In [ ]:
from pipeline.data_preprocessing import DataPreprocessing

array = DataPreprocessing().from_path(sample_path)

print("shape:", array.shape)
print("dtype:", array.dtype)
print(f"range: {array.min():.3f} - {array.max():.3f}")
print("\nModel expects (224, 224, 3) float32 in [0, 1].")